구글드라이브 연동

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


LSCP 코드

In [ ]:
!pip install pulp
!pip install geopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 77.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from geopy.distance import geodesic
from pulp import LpProblem, LpVariable, lpSum, LpMinimize, LpBinary, LpStatus

# 엑셀 파일 경로 지정
df = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/LSCP 데이터셋_최종.xlsx")
# 컬럼명 확인 (예: 위도, 경도라는 한글 컬럼이 있는 경우)

df.columns = [col.strip().replace(" ", "").replace("/", "") for col in df.columns]

# 필요한 컬럼만 사용
df = df[['이름', '위도', '경도', '수요후보구분', '기존존재여부']]

# ---------------------
# 2. 수요지 및 후보지 구분
# ---------------------
demand_df = df[df['수요후보구분'] == '수요지'].reset_index(drop=True)

candidate_df = df[df['수요후보구분'] == '후보지'].reset_index(drop=True)
must_include_df = candidate_df[candidate_df['기존존재여부'] == '기존'].reset_index(drop=True)
optional_candidate_df = candidate_df[candidate_df['기존존재여부'] != '기존'].reset_index(drop=True)

# 인덱스 정리
demand_points = demand_df[['이름', '위도', '경도']].copy()
must_facilities = must_include_df[['이름', '위도', '경도']].copy()
optional_facilities = optional_candidate_df[['이름', '위도', '경도']].copy()

# ---------------------
# 3. 커버리지 매트릭스 생성 (540m 이내)
# ---------------------
COVERAGE_RADIUS = 0.54  # km

# 후보지 전체 (기존 + 신규)
all_facilities = pd.concat([must_facilities, optional_facilities], ignore_index=True)

# 수요지 이름 목록
demand_names = demand_points['이름'].tolist()
facility_names = all_facilities['이름'].tolist()

# 커버리지 딕셔너리: 수요지 -> 커버 가능한 후보지 목록
coverage = {
    d: [] for d in demand_names
}

for i, d_row in demand_points.iterrows():
    d_name = d_row['이름']
    d_loc = (d_row['위도'], d_row['경도'])

    for j, f_row in all_facilities.iterrows():
        f_name = f_row['이름']
        f_loc = (f_row['위도'], f_row['경도'])

        if geodesic(d_loc, f_loc).km <= COVERAGE_RADIUS:
            coverage[d_name].append(f_name)

# ---------------------
# 4. LSCP 모델 정의
# ---------------------
model = LpProblem("LSCP", LpMinimize)

# 결정 변수: 후보지 선택 여부 (1: 선택, 0: 선택 안 함)
x = {f: LpVariable(f"x_{f}", cat=LpBinary) for f in facility_names}

# 반드시 포함해야 하는 후보지는 x = 1로 고정
for f in must_facilities['이름']:
    x[f].setInitialValue(1)
    x[f].fixValue()

# 목적 함수: 최소 후보지 수 선택
model += lpSum([x[f] for f in optional_facilities['이름']]), "Minimize Facility Count"

# 제약 조건: 모든 수요지는 최소 1개 이상 후보지에 의해 커버되어야 함
for d in demand_names:
    model += lpSum([x[f] for f in coverage[d]]) >= 1, f"Cover_{d}"

# ---------------------
# 5. 문제 풀이
# ---------------------
model.solve()

# 결과 출력
print("Status:", LpStatus[model.status])
selected_facilities = [f for f in facility_names if x[f].value() == 1]
print("선택된 후보지 수:", len(selected_facilities))
print("선택된 후보지:")
for f in selected_facilities:
    print(f)




from IPython.display import display

# 선택된 후보지만 추출
selected_df = all_facilities[all_facilities['이름'].isin(selected_facilities)].copy()


# 표 출력 (Jupyter에서 깔끔하게)
display(selected_df)

Status: Infeasible
선택된 후보지 수: 42
선택된 후보지:
점핑몬스터 구의점
리틀비틀 자양점
(주)퐁퐁플라워
더키즈 프리미엄 키즈카페
아이점프 군자점
하이하이
헬로방방(자양점)
서울형키즈카페 시립 팔각당점
서울형 키즈카페 자양4동 2호점(꾸미팡팡 놀이터)
서울형 키즈카페 자양4동 1호점(꾸미팡팡 놀이터)
서울형 키즈카페 시립 뚝섬자벌레점
서울형 키즈카페 광진구 중곡3동점(꾸미팡팡 놀이터)
새움터지역아동센터
해피엘지역아동센터
광진구10호점 우리동네키움센터
신양하늘꿈지역아동센터
꿈나래지역아동센터
희망샘지역아동센터
진성지역아동센터
광진구3호점 우리동네키움센터
자양 우리동네 키움센터(다함께 돌봄센터)
중곡지역아동센터
어린이나라지역아동센터
새빛지역아동센터
꿈 정함 우리동네 키움센터(다함께돌봄센터)
새날지역아동센터
경희지역아동센터
한사랑지역아동센터
광진구6호점 우리동네키움센터(중곡 정함)
광진구4호점 우리동네키움센터
광진구5호점 우리동네키움센터
광진구7호점 우리동네키움센터
광진구8호점 우리동네키움센터
행복한지역아동센터
광진구9호점 우리동네키움센터
영화어린이집
1
3
4
6
18
19


,이름,위도,경도
0,점핑몬스터 구의점,37.545499,127.088778
1,리틀비틀 자양점,37.533423,127.068805
2,(주)퐁퐁플라워,37.538123,127.072377
3,더키즈 프리미엄 키즈카페,37.559148,127.077316
4,아이점프 군자점,37.550770,127.070891
5,하이하이,37.557477,127.086381
6,헬로방방(자양점),37.535921,127.070528
7,서울형키즈카페 시립 팔각당점,37.549857,127.082471
8,서울형 키즈카페 자양4동 2호점(꾸미팡팡 놀이터),37.539160,127.069640
9,서울형 키즈카페 자양4동 1호점(꾸미팡팡 놀이터),37.539160,127.069640


In [ ]:
# 엑셀로 저장
selected_df.to_excel("/content/selected_facilities.xlsx", index=False)

# 다운로드
from google.colab import files
files.download("/content/selected_facilities.xlsx")


A(빨강)와 B(파랑)의 위치를 확인하는 지도시각화(수요지는 X)

In [ ]:
#####A와 B의 위치만 확인

import folium

# 지도 중심 좌표
center_lat = df['위도'].mean()
center_lon = df['경도'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

# 2. 기존 후보지 중 최종 선택된 것 (빨간색)
selected_must = must_facilities[must_facilities['이름'].isin(selected_facilities)]
for _, row in selected_must.iterrows():
    folium.CircleMarker(
        location=[row['위도'], row['경도']],
        radius=7,
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.9,
        popup=f"[기존 후보지] {row['이름']}"
    ).add_to(m)
print(len(selected_must))

# 3. 신규 후보지 중 최종 선택된 것 (파란색)
selected_new = optional_facilities[optional_facilities['이름'].isin(selected_facilities)]
for _, row in selected_new.iterrows():
    folium.CircleMarker(
        location=[row['위도'], row['경도']],
        radius=7,
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.9,
        popup=f"[선택된 신규 후보지] {row['이름']}"
    ).add_to(m)

print(len(selected_new))

# 지도 출력
m


35
7


In [ ]:
# 지도 저장
m.save("/content/selected_facilities_map.html")

# 다운로드
from google.colab import files
files.download("/content/selected_facilities_map.html")


A(빨강)와 B(파랑)으로 하고 수요지를 점 찍어서 확인

In [ ]:
import folium

# 기존 후보지 중 최종 선택된 것만 필터링
final_must = selected_df[selected_df['이름'].isin(must_facilities['이름'])]
final_optional = selected_df[~selected_df['이름'].isin(must_facilities['이름'])]

# 지도 중심 설정
center_lat = selected_df['위도'].mean()
center_lon = selected_df['경도'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

# 1. 수요지 (작은 검은 점)
for _, row in demand_df.iterrows():
    folium.CircleMarker(
        location=[row['위도'], row['경도']],
        radius=3,
        color='black',
        fill=True,
        fill_color='black',
        fill_opacity=0.6,
        popup=f"[수요지] {row['이름']}"
    ).add_to(m)

# 2. 최종 후보지 중 기존 후보지 (빨간 마커)
for _, row in final_must.iterrows():
    folium.Marker(
        location=[row['위도'], row['경도']],
        popup=f"[기존 후보지] {row['이름']}",
        icon=folium.Icon(color='red', icon='home', prefix='fa')
    ).add_to(m)

# 3. 최종 후보지 중 신규 후보지 (파란 마커)
for _, row in final_optional.iterrows():
    folium.Marker(
        location=[row['위도'], row['경도']],
        popup=f"[신규 후보지] {row['이름']}",
        icon=folium.Icon(color='blue', icon='plus', prefix='fa')
    ).add_to(m)

# 결과 지도 출력
m


In [ ]:
# 결과 지도 저장 및 다운로드
m.save("/content/final_facility_map.html")

from google.colab import files
files.download("/content/final_facility_map.html")


A(빨강)와 B(파랑)을 찍고 각 후보지가 커버하는 수요지를 면적으로 나타내기

In [ ]:
import folium

# 최종 선택된 기존 후보지, 신규 후보지 구분
final_must = selected_df[selected_df['이름'].isin(must_facilities['이름'])]
final_optional = selected_df[~selected_df['이름'].isin(must_facilities['이름'])]

# 지도 중심 좌표 설정
center_lat = df['위도'].mean()
center_lon = df['경도'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

# 1. 수요지 (540m 회색 원)
for _, row in demand_df.iterrows():
    folium.Circle(
        location=[row['위도'], row['경도']],
        radius=540,  # 단위: 미터
        color='black',
        fill=True,
        fill_color='gray',
        fill_opacity=0.3,
        popup=f"[수요지] {row['이름']}"
    ).add_to(m)

# 2. 최종 선택된 기존 후보지 (빨간색 마커)
for _, row in final_must.iterrows():
    folium.Marker(
        location=[row['위도'], row['경도']],
        popup=f"[기존 후보지] {row['이름']}",
        icon=folium.Icon(color='red', icon='home', prefix='fa')
    ).add_to(m)

# 3. 최종 선택된 신규 후보지 (파란색 마커)
for _, row in final_optional.iterrows():
    folium.Marker(
        location=[row['위도'], row['경도']],
        popup=f"[신규 후보지] {row['이름']}",
        icon=folium.Icon(color='blue', icon='plus', prefix='fa')
    ).add_to(m)

# 지도 출력
m


B 후보지에 대해서 자신을 제외한 나머지 49개 후보지와의 거리

In [ ]:
###데이터셋 1번

import pandas as pd
from geopy.distance import geodesic
from google.colab import files

# 'b' 후보지: 기존 아닌 후보지 중 선택된 것들 (8개)
b_facilities = selected_df[~selected_df['이름'].isin(must_facilities['이름'])].copy()

# 전체 후보지에서 각 b 후보지를 하나씩 제외하고 비교 대상 만들기
rows = []

for i, b_row in b_facilities.iterrows():
    b_name = b_row['이름']
    b_loc = (b_row['위도'], b_row['경도'])

    # 자기 자신(b 후보지)만 제외한 나머지 후보지 49개
    others = selected_df[selected_df['이름'] != b_name].copy()

    distances = []

    for j, other_row in others.iterrows():
        other_name = other_row['이름']
        other_loc = (other_row['위도'], other_row['경도'])

        dist = geodesic(b_loc, other_loc).km
        distances.append((other_name, dist))

    # 가까운 5개 추출
    distances = sorted(distances, key=lambda x: x[1])[:5]

    for close_name, close_dist in distances:
        rows.append({
            '새후보지명': b_name,
            '거리가까운후보지': close_name,
            '거리': round(close_dist, 4)
        })

# 결과 DataFrame 생성
result_df = pd.DataFrame(rows)

# 엑셀로 저장
excel_path = '새후보지_근접후보지_거리표_1번최종.xlsx'
result_df.to_excel(excel_path, index=False)

# 파일 다운로드
files.download(excel_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

50개의 후보지별로 수요지리스트

In [ ]:
#####데이터셋 2번 후보지의 수요지리스트

import pandas as pd
from geopy.distance import geodesic
from google.colab import files

COVERAGE_RADIUS = 0.54  # km

rows = []

# 수요지 목록
for i, f_row in selected_df.iterrows():
    f_name = f_row['이름']
    f_loc = (f_row['위도'], f_row['경도'])

    covered_demands = []

    for j, d_row in demand_df.iterrows():
        d_name = d_row['이름']
        d_loc = (d_row['위도'], d_row['경도'])

        if geodesic(f_loc, d_loc).km <= COVERAGE_RADIUS:
            covered_demands.append(d_name)

    rows.append({
        '후보지명': f_name,
        '수요지리스트': ','.join(covered_demands),
        '수요지개수': len(covered_demands)
    })

# 결과 DataFrame
coverage_df = pd.DataFrame(rows)

# 엑셀 저장
excel_path = '후보지별_커버수요지_정보_2번최종.xlsx'
coverage_df.to_excel(excel_path, index=False)

# 다운로드
files.download(excel_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

50개의 후보지별 수요지별 겹치는 후보지리스트

In [ ]:
#####데이터셋 3번

import pandas as pd
from geopy.distance import geodesic
from google.colab import files

COVERAGE_RADIUS = 0.54  # km

# Step 1: 후보지별로 커버하는 수요지 리스트 만들기
facility_coverage = {}

for i, f_row in selected_df.iterrows():
    f_name = f_row['이름']
    f_loc = (f_row['위도'], f_row['경도'])

    covered_demands = []

    for j, d_row in demand_df.iterrows():
        d_name = d_row['이름']
        d_loc = (d_row['위도'], d_row['경도'])

        if geodesic(f_loc, d_loc).km <= COVERAGE_RADIUS:
            covered_demands.append(d_name)

    facility_coverage[f_name] = covered_demands

# Step 2: 수요지 기준으로 어떤 후보지가 커버하고 있는지 매핑
demand_to_facilities = {}

for f_name, demand_list in facility_coverage.items():
    for d_name in demand_list:
        if d_name not in demand_to_facilities:
            demand_to_facilities[d_name] = []
        demand_to_facilities[d_name].append(f_name)

# Step 3: 최종 테이블 구성
rows = []

for f_name, demand_list in facility_coverage.items():
    for d_name in demand_list:
        overlapping = [other for other in demand_to_facilities[d_name] if other != f_name]
        if overlapping:
            overlapping_str = ','.join(map(str, overlapping))  # 수정된 부분
        else:
            overlapping_str = '(없음)'

        rows.append({
            '후보지명': f_name,
            '겹치는수요지명': d_name,
            '겹치는후보지': overlapping_str
        })

# DataFrame으로 변환
overlap_df = pd.DataFrame(rows)

# 엑셀로 저장 후 다운로드
excel_path = '후보지별_겹치는_수요지정보_3번최종.xlsx'
overlap_df.to_excel(excel_path, index=False)
files.download(excel_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

수요지별로 겹치는 후보지리스트

In [ ]:
######추가 데이터셋 수요지별 겹치는 후보지명
import pandas as pd
from google.colab import files

# 수요지 → 커버하는 후보지 리스트를 기반으로 테이블 구성
rows = []

for d_name, facilities in demand_to_facilities.items():
    rows.append({
        '겹치는수요지명': d_name,
        '겹치는후보지명': ','.join(map(str, facilities)),
        '후보지개수': len(facilities)
    })

# DataFrame으로 변환
demand_overlap_df = pd.DataFrame(rows)

# 엑셀로 저장
excel_path = '수요지별_겹치는_후보지정보_최종.xlsx'
demand_overlap_df.to_excel(excel_path, index=False)

# 파일 다운로드
files.download(excel_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

B에 대해서 이름, 주소, 위경도, 서비스분류명을 담은 파일

In [ ]:
####추가 데이터셋:   b에 대해서만 이름, 주소, 위도, 경도 , 서비스분류명을 담은 엑셀
import pandas as pd
from google.colab import files

newdf = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/LSCP 데이터셋_최종.xlsx")
# 컬럼명 확인 (예: 위도, 경도라는 한글 컬럼이 있는 경우)
newdf.columns = [col.strip().replace(" ", "").replace("/", "") for col in newdf.columns]

# b 후보지 이름 목록
b_names = b_facilities['이름'].tolist()

# df에서 이름이 b 후보지에 해당하는 행만 추출
b_info_df = newdf[newdf['이름'].isin(b_names)][['이름', '주소', '위도', '경도', '서비스분류명']].copy()

# 엑셀 저장 및 다운로드
excel_path = 'b후보지_정보_최종.xlsx'
b_info_df.to_excel(excel_path, index=False)
files.download(excel_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>